In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

In [ ]:
y = np.log1p(train["SalePrice"])
X = train.drop(columns = ["SalePrice" , "Id"])
X_test = test.drop(columns = ["Id"])

test_ids = test["Id"]

print(X.shape, X_test.shape, y.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
num_cols = X.select_dtypes(include =["int64", "float64"]).columns
cat_cols = X.select_dtypes(include = ["object"]).columns

numeric_pipe = Pipeline(steps = [("imputer" , SimpleImputer(strategy = "median"))])

categorical_pipe =  Pipeline(steps = [("imputer", SimpleImputer(strategy = "most_frequent")),
                                     ("ohe", OneHotEncoder(handle_unknown = "ignore", sparse_output = False))])

preprocess = ColumnTransformer(transformers = [("num", numeric_pipe, num_cols),("cats", categorical_pipe, cat_cols)]) 

In [ ]:
from sklearn.linear_model import Ridge
model = Ridge(alpha = 5.0 , random_state = 42)

In [ ]:
pipe = Pipeline(steps = [("prep" , preprocess), ("model", model)])

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_root_mean_squared_error")

print(f"CV RMSLE: {scores.mean():.5f} (+/- {scores.std():.5f})")


In [ ]:
pipe.fit(X,y)
pred_log = pipe.predict(X_test)
pred = np.expm1(pred_log)

In [ ]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": pred
})
submission.to_csv("submission.csv", index=False)